# AI Mathematical Olympiad - Progress Prize 3

### Architecture:
####   Qwen3-32B in thinking mode (built-in chain-of-thought reasoning)
####   - Tool-Integrated Reasoning (model writes + executes Python mid-solve)
####   - Majority voting across 8 samples (optimal for 32B in 5hr budget)
####   - Adaptive time buffer (unused time from easy problems - hard problems)
####   - Robust answer extraction with 3 fallback levels

# Block Sklearn

In [6]:
import sys

# Block sklearn entirely before transformers tries to import it
_BLOCKED = [
    'sklearn', 'sklearn.metrics', 'sklearn.utils',
    'sklearn.utils.murmurhash', 'sklearn.base',
]
for _mod in _BLOCKED:
    sys.modules[_mod] = None  # type: ignore

print("sklearn blocked — numpy conflict prevented")

sklearn blocked — numpy conflict prevented


# Imports

In [7]:
import os
import re
import time
import subprocess
import tempfile
import warnings
from collections import Counter

import torch
import pandas as pd
import polars as pl
import kaggle_evaluation.aimo_3_inference_server

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f" Imports complete")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

 Imports complete
  PyTorch: 2.6.0+cu124
  CUDA available: True
  GPU: NVIDIA H100 80GB HBM3
  VRAM: 85.0 GB


# Find Model Path

In [8]:
def find_model_path(search_root: str = "/kaggle/input") -> str:
    """
    Finds the directory containing .safetensors model weight files.
    Returns the path so we can load from it directly.
    """
    for root, dirs, files in os.walk(search_root):
        if any(f.endswith(".safetensors") for f in files):
            print(f"Model weights found at:\n  {root}")
            return root
    raise FileNotFoundError(
        "\n No model weights found under /kaggle/input\n"
        "  Fix: sidebar → Add Input - Models - search Qwen/Qwen3-32B - Add"
    )

MODEL_PATH = find_model_path()

Model weights found at:
  /kaggle/input/models/deepseek-ai/deepseek-r1/transformers/deepseek-r1-distill-qwen-32b/2


# Configuration

### Tuned specifically for Qwen3-32B on a single H100 (80GB) within 5hr budget.

In [9]:
# Model settings (from Qwen3 official math documentation)
TEMPERATURE     = 0.6     # Official Qwen3 recommendation for math tasks
TOP_P           = 0.95    # Official Qwen3 recommendation
TOP_K           = 20      # Official Qwen3 recommendation
MAX_NEW_TOKENS  = 3000    # Tokens generated per sample (thinking + answer)

# Voting settings
# 5 samples is safe for 32B within 5hrs. Each sample is ~50s on H100.
# 55 problems × 5 samples × 50s = 4.6hrs — fits with buffer.
N_SAMPLES       = 5

# Stop early if this many samples agree (saves time for hard problems)
EARLY_STOP_AT   = 3       # Stop after this many samples agree

# Time budget per problem (seconds)
BASE_TIME       = 300     # Base allocation per problem
MAX_BUFFER      = 180     # Max extra seconds from buffer pool
CODE_TIMEOUT    = 15      # Kill code execution after this many seconds

print("Configuration loaded")
print(f"  Model:          Qwen3-32B (thinking mode)")
print(f"  Samples:        {N_SAMPLES} per problem")
print(f"  Max tokens:     {MAX_NEW_TOKENS} per sample")
print(f"  Temperature:    {TEMPERATURE}")
print(f"  Time budget:    {BASE_TIME}s base + up to {MAX_BUFFER}s buffer")

Configuration loaded
  Model:          Qwen3-32B (thinking mode)
  Samples:        5 per problem
  Max tokens:     3000 per sample
  Temperature:    0.6
  Time budget:    300s base + up to 180s buffer


# Load Model

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    padding_side="left",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit quantization...")
t0 = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="cuda:0",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()

print(f" Model loaded in {time.time()-t0:.1f}s")
print(f"  Device: {next(model.parameters()).device}")
allocated = torch.cuda.memory_allocated(0) / 1e9
print(f"  GPU memory used: {allocated:.1f} GB / 80.0 GB")

Loading tokenizer...
Loading model in 4-bit quantization...


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

✓ Model loaded in 53.4s
  Device: cuda:0
  GPU memory used: 19.2 GB / 80.0 GB


# Code Execution Sandbox

In [11]:
def execute_code(code: str, timeout: int = CODE_TIMEOUT) -> str:
    """
    Runs a Python snippet in a subprocess and returns its output.
    Output is capped at 1500 chars to avoid flooding the model context.
    """
    preamble = (
        "import sys, math, itertools, functools\n"
        "from fractions import Fraction\n"
        "from collections import Counter, defaultdict\n"
        "from sympy import *\n"
        "from sympy.ntheory import factorint, isprime, primerange, totient\n\n"
    )
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False, dir="/tmp"
    ) as f:
        f.write(preamble + code)
        fpath = f.name

    try:
        proc = subprocess.run(
            ["python3", fpath],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        raw = (proc.stdout + proc.stderr).strip()
        return raw[:1500] if raw else "[No output]"
    except subprocess.TimeoutExpired:
        return f"[Code timed out after {timeout}s — simplify the computation]"
    except Exception as e:
        return f"[Execution error: {e}]"
    finally:
        try:
            os.unlink(fpath)
        except OSError:
            pass

# Verify sandbox works before touching competition problems
_test = execute_code("print(sum(factorint(720).values()))")
assert _test.strip() != "", f"Sandbox broken: {_test}"
print(f"Code sandbox working — test output: {_test.strip()}")

Code sandbox working — test output: 7


# System Prompt

### Designed specifically for Qwen3-32B thinking mode + TIR.
### Key: /think tag activates deep reasoning. Code blocks get executed.

In [12]:
SYSTEM_PROMPT = """\
/think

You are an elite mathematical olympiad solver competing at IMO level.
You have deep expertise in algebra, combinatorics, geometry, and number theory.

## How to solve
1. Read the problem carefully and identify exactly what is being asked.
2. Think rigorously step by step — never skip logical steps.
3. When you need to compute, verify, or brute-force something, write Python
   code in a ```python block. It will be executed and output shown to you.
4. Use that output to validate or continue your reasoning.
5. Double-check your answer before finalizing.

## Code you can use
sympy, math, itertools, fractions are pre-imported in all code blocks.
Example:
```python
from sympy import factorint
print(factorint(360))
```

## Final answer format
- Must be a single integer between 0 and 99999 inclusive.
- End your response with EXACTLY this — no exceptions:
\\boxed{YOUR_INTEGER}\
"""

In [13]:
def build_prompt(problem_text: str) -> str:
    """
    Formats a problem into Qwen3 chat format with thinking mode active.
    Uses the tokenizer's built-in chat template for correct formatting.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Solve this completely:\n\n{problem_text}"},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

print("Prompt builder ready")

Prompt builder ready


# Answer Extraction

In [14]:
def extract_answer(text: str) -> int | None:
    """
    Extracts integer answer from model output.

    Priority:
      1. \\boxed{N}          — model used the required format
      2. "answer is/= N"    — model stated it in natural language
      3. Last integer       — last-resort fallback
    Returns None if nothing found.
    """
    # Remove <think>...</think> block — search only the final answer section
    clean = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    search = clean if clean else text

    # Method 1: \boxed{N} — highest confidence, required format
    boxed = re.findall(r'\\boxed\{\s*(\d+)\s*\}', search)
    if boxed:
        return int(boxed[-1]) % 100000

    # Method 2: explicit answer statement
    stated = re.findall(
        r'(?:(?:final\s+)?answer\s*(?:is|=|:)\s*)(\d+)',
        search, re.IGNORECASE
    )
    if stated:
        return int(stated[-1]) % 100000

    # Method 3: last integer in response
    all_ints = re.findall(r'\b(\d{1,6})\b', search)
    if all_ints:
        return int(all_ints[-1]) % 100000

    return None

print("Answer extraction ready")

Answer extraction ready


# Single-Sample Tir Solver

### One full attempt: generate - detect code - execute - feed back - repeat.
### Up to 5 rounds of code execution per sample.

In [15]:
def generate(prompt: str) -> str:
    """
    Runs one forward pass through Qwen3-32B.
    Returns only the newly generated tokens as a string.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=8192,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Slice off the input tokens — we only want what was generated
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def solve_once(problem_text: str) -> int:
    """
    One full solve attempt with Tool-Integrated Reasoning.
    The model can write ```python blocks up to 4 times.
    Each block is executed and output fed back before continuing.
    Returns integer answer (0-99999).
    """
    prompt      = build_prompt(problem_text)
    accumulated = ""

    for round_idx in range(4):
        chunk       = generate(prompt + accumulated)
        accumulated += chunk

        # Check if the model wrote a code block
        code_match = re.search(r'```python\n(.*?)(?:```|$)', chunk, re.DOTALL)
        if code_match:
            code   = code_match.group(1).strip()
            result = execute_code(code)
            print(f"    [code block {round_idx+1}] output: {result[:80].strip()}")
            # Append output so model sees it in the next round
            accumulated += f"\n```output\n{result}\n```\n"
            continue

        # No code written — check for final answer
        answer = extract_answer(accumulated)
        if answer is not None:
            return answer
        break  # Model finished but no answer found

    # Final scan of everything generated
    answer = extract_answer(accumulated)
    return answer if answer is not None else 0

print("Single-sample TIR solver ready")

Single-sample TIR solver ready


# Majority Vote with Adaptive Time Budget

### Runs N_SAMPLES attempts, returns most common answer.
### Unused time from fast problems flows into a shared buffer for hard ones.
### This is the exact time management strategy used by the AIMO2 winners.

In [16]:
_time_buffer: float = 0.0  # Shared pool of unused seconds

def solve_problem(problem_text: str, problem_id: str = "?") -> int:
    """
    Full solver: runs up to N_SAMPLES attempts, returns majority answer.
    Manages time budget across all problems automatically.
    """
    global _time_buffer

    budget  = BASE_TIME + min(_time_buffer, MAX_BUFFER)
    start   = time.time()
    answers = []

    print(f"\n{'='*55}")
    print(f"Problem {problem_id} | Budget: {budget:.0f}s "
          f"(base {BASE_TIME}s + {min(_time_buffer, MAX_BUFFER):.0f}s buffer)")
    print(f"{'='*55}")

    for i in range(N_SAMPLES):
        elapsed = time.time() - start
        if elapsed >= budget:
            print(f"  Time budget used ({elapsed:.0f}s) — stopping at {i} samples")
            break

        try:
            ans = solve_once(problem_text)
            answers.append(ans)
            print(f"  Sample {i+1}/{N_SAMPLES}: {ans}")

            # Early stop: if EARLY_STOP_AT samples all agree, no need to continue
            if len(answers) >= EARLY_STOP_AT:
                top, cnt = Counter(answers).most_common(1)[0]
                if cnt >= EARLY_STOP_AT:
                    print(f" Early stop — {cnt}/{len(answers)} samples agree on {top}")
                    _time_buffer += max(0.0, BASE_TIME - (time.time() - start))
                    print(f"  Buffer updated: {_time_buffer:.0f}s total")
                    return top

        except Exception as e:
            print(f" Sample {i+1} failed: {type(e).__name__}: {e}")
            continue

    # Add unused time to buffer for the next problem
    used          = time.time() - start
    leftover      = max(0.0, BASE_TIME - used)
    _time_buffer += leftover
    print(f"  Buffer updated: +{leftover:.0f}s → {_time_buffer:.0f}s total")

    if not answers:
        print(" All samples failed — returning 0")
        return 0

    best, cnt = Counter(answers).most_common(1)[0]
    confidence = cnt / len(answers)
    print(f"   Final answer: {best}  "
          f"({cnt}/{len(answers)} votes, {confidence:.0%} confidence)")
    return best

print("Majority vote solver ready")

Majority vote solver ready


# Competition Model Wrapper
### Preserves the exact template structure the AIMO3 inference server expects.

In [17]:
class Model:
    """
    Wraps the solver for the AIMO3 inference server.
    Tracks problem count and delegates to solve_problem().
    """
    def __init__(self):
        self._problem_num = 0
        self._ready       = False

    def load(self):
        # Model is already loaded globally above — just mark as ready
        self._ready = True
        print("Competition model interface ready")

    def predict(self, problem: str) -> int:
        if not self._ready:
            self.load()
        self._problem_num += 1
        return solve_problem(problem, problem_id=str(self._problem_num))


model_wrapper = Model()


def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame | pd.DataFrame:
    """
    AIMO3 competition entry point.
    Called once per problem by the inference server.
    Must return a DataFrame with columns ['id', 'answer'].
    """
    id_val       = id_.item(0)
    problem_text = problem.item(0)
    answer       = model_wrapper.predict(problem_text)
    return pl.DataFrame({"id": [id_val], "answer": [answer]})

print("Competition interface ready")

Competition interface ready


# Run
### Locally: tests against sample problems in /kaggle/input/
### On submission: switches to inference_server.serve() automatically.

In [18]:
model = model.to("cuda")
print(f"Model device: {next(model.parameters()).device}")

Model device: cuda:0


In [19]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(
    predict
)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    # LIVE COMPETITION RUN
    inference_server.serve()
else:
    # LOCAL TEST RUN
    print("\n" + "="*55)
    print("LOCAL TEST — running against sample problems")
    print("="*55 + "\n")
    inference_server.run_local_gateway(
        ("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",)
    )


LOCAL TEST — running against sample problems

Competition model interface ready

Problem 1 | Budget: 300s (base 300s + 0s buffer)
    [code block 1] output: 0
  Sample 1/5: 0
    [code block 1] output: 0
  Sample 2/5: 0
    [code block 1] output: 0
  Sample 3/5: 0
 Early stop — 3/3 samples agree on 0
  Buffer updated: 165s total

Problem 2 | Budget: 465s (base 300s + 165s buffer)
    [code block 1] output: True
  Sample 1/5: 0


KeyboardInterrupt: 

I0000 00:00:1773096447.443890     490 chttp2_transport.cc:1335] ipv6:%5B::1%5D:50051: Got goaway [2] err=UNAVAILABLE:GOAWAY received; Error code: 2; Debug Text: Cancelling all calls {http2_error:2, grpc_status:14}


    [code block 1] output: True
  Sample 2/5: 0
  Sample 3/5: 0
 Early stop — 3/3 samples agree on 0
  Buffer updated: 334s total
